# In-vivo multi-seed summary

Loads the per-seed result CSVs produced by `scripts/invivo_multi_seed.py`
(`results/invivo_multiseed_{cellina,cellina-gat,spatialprop}.csv`) and reports
mean ± std **over seeds** for each (model, type, baseline), analogous to the
single-run `summary_fmt` table in `pfish_analysis.ipynb` §4.3.

Note: the `mean` baseline is model-agnostic (no model calls at all) and is only
ever saved under `model == 'cellina'` by the training script, so it appears once
in this summary rather than once per model.

In [1]:
import glob
import os

import numpy as np
import pandas as pd

RESULTS_DIR = "../../results"
METRICS = ["Pearson", "Precision", "E-distance", "RMSE_LFC"]

## Load and combine

In [2]:
csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, "invivo_multiseed_*.csv")))
print("found:\n " + "\n ".join(csv_paths))

df = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"\n{len(df)} rows | models: {sorted(df['baseline'].unique())}")
df.groupby("baseline")["seed"].nunique().rename("n_seeds").to_frame()

found:
 ../../results/invivo_multiseed_cellina.csv

162 rows | models: ['Cellina-base', 'Cellina-base-random', 'mean']


,n_seeds
baseline,
Cellina-base,3
Cellina-base-random,3
mean,3


In [3]:
df

,KO,type,baseline,Pearson,Precision,E-distance,RMSE_LFC,seed,model
0,MAP2K2,full,Cellina-base,0.817882,0.9,3.704329,0.550358,0,cellina
1,MAP2K2,KO-specific,Cellina-base,0.894314,1.0,3.669840,0.435427,0,cellina
2,IRAK1,full,Cellina-base,0.507100,0.8,4.384122,0.563590,0,cellina
3,IRAK1,KO-specific,Cellina-base,0.606950,0.8,4.408971,0.372227,0,cellina
4,IRF7,full,Cellina-base,0.879831,0.9,3.867988,0.280424,0,cellina
...,...,...,...,...,...,...,...,...,...
157,NFKBIA,KO-specific,mean,0.448364,0.6,0.149922,0.585901,2,cellina
158,LBP,full,mean,0.724704,0.8,0.033160,0.412855,2,cellina
159,LBP,KO-specific,mean,0.282640,0.6,0.066309,0.378432,2,cellina
160,MAP2K3,full,mean,0.558055,0.9,0.096456,0.469386,2,cellina


## Per-seed summary

Average each metric over the 9 KOs within a (model, seed, type, baseline) --
one score per training run, the same quantity `pfish_analysis.ipynb`'s single-run
`summary_fmt` reports (just not yet aggregated across seeds).

In [4]:
per_seed = df.groupby(["seed", "type", "baseline"])[METRICS].mean().reset_index()
per_seed

,seed,type,baseline,Pearson,Precision,E-distance,RMSE_LFC
0,0,KO-specific,Cellina-base,0.311139,0.711111,3.938490,0.413531
1,0,KO-specific,Cellina-base-random,0.028063,0.488889,3.861616,0.427254
2,0,KO-specific,mean,-0.009052,0.544444,0.143202,0.427040
3,0,full,Cellina-base,0.570246,0.777778,3.909132,0.470999
4,0,full,Cellina-base-random,0.177850,0.593333,3.871998,0.553791
5,0,full,mean,0.365141,0.766667,0.135579,0.519615
6,1,KO-specific,Cellina-base,0.235709,0.633333,4.272027,0.446487
7,1,KO-specific,Cellina-base-random,0.031446,0.520000,4.250787,0.458813
8,1,KO-specific,mean,0.046099,0.466667,0.130473,0.464918
9,1,full,Cellina-base,0.562250,0.744444,4.219859,0.515413


## Summary: mean ± std over seeds

In [5]:
# numeric mean/std, for programmatic use
summary = per_seed.groupby(["type", "baseline"])[METRICS].agg(["mean", "std"])
summary

Pearson           Precision            \
                                     mean       std      mean       std   
type        baseline                                                      
KO-specific Cellina-base         0.268459  0.038683  0.670370  0.039021   
            Cellina-base-random -0.011502  0.071478  0.497037  0.020164   
            mean                 0.058448  0.074447  0.518519  0.044905   
full        Cellina-base         0.537700  0.049608  0.755556  0.019245   
            Cellina-base-random  0.160231  0.310290  0.568889  0.056350   
            mean                 0.533506  0.165230  0.781481  0.025660   

                                E-distance            RMSE_LFC            
                                      mean       std      mean       std  
type        baseline                                                      
KO-specific Cellina-base          4.108512  0.166863  0.431894  0.016798  
            Cellina-base-random   4.061338  0.194788  0.454816  0.025796  
            mean                  0.134857  0.007230  0.450937  0.020795  
full        Cellina-base          4.072183  0.155933  0.489538  0.023098  
            Cellina-base-random   4.082622  0.207664  0.572671  0.022965  
            mean                  0.129230  0.005662  0.516246  0.005147

In [6]:
n_seeds = per_seed.groupby(["type", "baseline"])["seed"].nunique().rename("n_seeds")

summary_fmt = per_seed.groupby(["type", "baseline"])[METRICS].agg(
    lambda x: f"{x.mean():.2f} \u00b1 {x.std():.2f}"
)
summary_fmt = summary_fmt.join(n_seeds)
summary_fmt

Pearson    Precision   E-distance  \
type        baseline                                                      
KO-specific Cellina-base          0.27 ± 0.04  0.67 ± 0.04  4.11 ± 0.17   
            Cellina-base-random  -0.01 ± 0.07  0.50 ± 0.02  4.06 ± 0.19   
            mean                  0.06 ± 0.07  0.52 ± 0.04  0.13 ± 0.01   
full        Cellina-base          0.54 ± 0.05  0.76 ± 0.02  4.07 ± 0.16   
            Cellina-base-random   0.16 ± 0.31  0.57 ± 0.06  4.08 ± 0.21   
            mean                  0.53 ± 0.17  0.78 ± 0.03  0.13 ± 0.01   

                                    RMSE_LFC  n_seeds  
type        baseline                                   
KO-specific Cellina-base         0.43 ± 0.02        3  
            Cellina-base-random  0.45 ± 0.03        3  
            mean                 0.45 ± 0.02        3  
full        Cellina-base         0.49 ± 0.02        3  
            Cellina-base-random  0.57 ± 0.02        3  
            mean                 0.52 ± 0.01        3